In [ ]:
!pip install catboost

In [ ]:
import os
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from scipy.stats import randint, uniform, loguniform
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import matthews_corrcoef

from imblearn.ensemble import (
    BalancedRandomForestClassifier,
    EasyEnsembleClassifier
)

from sklearn.model_selection import(
    StratifiedKFold,
    StratifiedGroupKFold,
    RandomizedSearchCV,
    cross_val_predict
)

from utils.models import (
    get_models,
    get_base_models,
    get_age_baseline
)

from utils.evaluation import evaluate_model

from utils.optimization import(
    get_param_dist,
    get_search_iter
)

from utils.pipeline import(
    create_pipeline,
    # find_best_threshold,
    save_fold_predictions
)
from itertools import chain

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.feature_selection import RFECV
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.over_sampling import SMOTENC, SMOTE

from sklearn.ensemble import RandomForestClassifier

In [ ]:
models = get_models()
base_models = get_base_models()
age_baseline = get_age_baseline()

In [3]:
import pandas as pd
ds_raw = pd.read_csv("/content/DiabeticCKD_dataset.csv")
ds_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 26 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Patient ID                                        4000 non-null   int64  
 1   Gender (M-male, F-female)                         4000 non-null   object 
 2   Job (1-normal, 2-intermediate, 3-heavy)           4000 non-null   int64  
 3   Family Background of Diabetes (1-yes, 0-no)       4000 non-null   int64  
 4   Height (cm)                                       4000 non-null   float64
 5   Diabetic Year                                     4000 non-null   int64  
 6   Age                                               4000 non-null   int64  
 7   Average Age                                       4000 non-null   int64  
 8   Weight (kg)                                       4000 non-null   int64  
 9   Average Weight (kg)

In [4]:
ds_raw.columns = ['Patient_ID', 'Gender',
       'Job', 'Family_Background_of_Diabetes', 'Height',
       'Diabetic_Year', 'Age', 'Average_Age', 'Weight',
       'Average_Weight', 'BMI', 'Follow_suggested_Diet',
       'Take_Medicine_for_Diabetes',
       'Take_Insulin', 'Hypertension',
       'Heart_Disease', 'Sleep',
       'Water_Consumption',
       'Smoke', 'Zarda_Betel_Leaf',
       'Walk_Regularly', 'Urination_Properly',
       'Urinary_Infection', 'Pain_killer',
       'Calorie_Intake', 'CKD']

In [5]:
ds_raw['Gender'] = ds_raw['Gender'].map({
    'F': 0,
    'M': 1
})

In [6]:
ds_raw["Current_BMI"] = (
    ds_raw["Weight"] /
    (ds_raw["Height"]/100)**2
)

In [10]:
continuous_features = [
    "Age",
    "Average_Age",
    "Height",
    "Weight",
    "Average_Weight",
    "BMI",
    "Diabetic_Year",
    "Calorie_Intake"
]
target = 'CKD'
ds_raw.drop("Patient_ID", axis=1, inplace=True)

In [11]:
ds_raw["BMI_Diabetes"] = (ds_raw["Current_BMI"]*ds_raw["Diabetic_Year"])
ds_raw["Age_Diabetes"] = (ds_raw["Age"]*ds_raw["Diabetic_Year"])
ds_raw["Weight_Diff"] = (ds_raw["Weight"]-ds_raw["Average_Weight"])
ds_raw["Age_Diff"] = (ds_raw["Age"]-ds_raw["Average_Age"])
ds_raw["BMI_per_Year"]=(ds_raw["Current_BMI"]/(ds_raw["Diabetic_Year"]+1))
ds_raw["Lifestyle_Score"]=(ds_raw["Walk_Regularly"]+ds_raw["Follow_suggested_Diet"]+
                           ds_raw["Sleep"]+ds_raw["Water_Consumption"])
ds_raw["Risk_Score"]=(ds_raw["Hypertension"]+ds_raw["Heart_Disease"]+
                      ds_raw["Smoke"]+ds_raw["Family_Background_of_Diabetes"])
ds_raw["Treatment_Score"]=(ds_raw["Take_Medicine_for_Diabetes"]+ds_raw["Take_Insulin"])

In [12]:
ds_raw.columns

Index(['Gender', 'Job', 'Family_Background_of_Diabetes', 'Height',
       'Diabetic_Year', 'Age', 'Average_Age', 'Weight', 'Average_Weight',
       'BMI', 'Follow_suggested_Diet', 'Take_Medicine_for_Diabetes',
       'Take_Insulin', 'Hypertension', 'Heart_Disease', 'Sleep',
       'Water_Consumption', 'Smoke', 'Zarda_Betel_Leaf', 'Walk_Regularly',
       'Urination_Properly', 'Urinary_Infection', 'Pain_killer',
       'Calorie_Intake', 'CKD', 'Current_BMI', 'BMI_Diabetes', 'Age_Diabetes',
       'Weight_Diff', 'Age_Diff', 'BMI_per_Year', 'Lifestyle_Score',
       'Risk_Score', 'Treatment_Score'],
      dtype='object')

In [13]:
ds_raw.to_csv("DiabeticCKD_engineered.csv", index=False)

In [15]:
ds2 = pd.read_csv("/content/DiabeticCKD_engineered.csv")
group_ds = pd.read_csv("/content/Patient_Groups.csv")

group = group_ds["Patient_ID"]
ds2_X = ds2.drop("CKD", axis=1)
ds2_y = ds2["CKD"]

In [16]:
ds2_X.columns

Index(['Gender', 'Job', 'Family_Background_of_Diabetes', 'Height',
       'Diabetic_Year', 'Age', 'Average_Age', 'Weight', 'Average_Weight',
       'BMI', 'Follow_suggested_Diet', 'Take_Medicine_for_Diabetes',
       'Take_Insulin', 'Hypertension', 'Heart_Disease', 'Sleep',
       'Water_Consumption', 'Smoke', 'Zarda_Betel_Leaf', 'Walk_Regularly',
       'Urination_Properly', 'Urinary_Infection', 'Pain_killer',
       'Calorie_Intake', 'Current_BMI', 'BMI_Diabetes', 'Age_Diabetes',
       'Weight_Diff', 'Age_Diff', 'BMI_per_Year', 'Lifestyle_Score',
       'Risk_Score', 'Treatment_Score'],
      dtype='object')

In [ ]:
#### Dataset 2 - Diabetic CKD Dataset ####
outer_cv_d2 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_cv_d2 = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)


In [ ]:
def tune_model(model_name, pipeline, X_train, y_train, inner_cv, scoring, groups=None):

  if model_name in ["Majority Dummy", "Stratified Dummy"]:
    pipeline.fit(X_train, y_train)
    return pipeline, {}, None

  ### RandomizedSearchCV ###
  random_search = RandomizedSearchCV(

      estimator=pipeline,
      param_distributions=get_param_dist(model_name),
      n_iter=get_search_iter(model_name),
      scoring=scoring,
      cv=inner_cv,
      random_state=RANDOM_STATE,
      n_jobs=-1,
      refit=True
    )

  random_search.fit(
          X_train,
          y_train,
          groups=groups
  )

  ### Best Estimator ###
  best_pipeline = random_search.best_estimator_
  best_params = random_search.best_params_
  best_score = random_search.best_score_

  return best_pipeline, best_params, best_score

In [ ]:
def select_best_threshold(model_name, best_param, X_train, y_train, inner_cv, groups=None):
  ### Generate inner out-of-fold probabilities ###
  inner_probabilities = np.zeros(len(y_train))

  # Generate inner folds
  if groups is None:
      splits = inner_cv.split(X_train, y_train)
  else:
      splits = inner_cv.split(X_train, y_train, groups)

  for inner_train_idx, inner_valid_idx in splits:

        # Split inner fold
        X_inner_train = X_train.iloc[inner_train_idx]
        y_inner_train = y_train.iloc[inner_train_idx]

        X_inner_valid = X_train.iloc[inner_valid_idx]

        # Create a fresh pipeline
        if model_name in ["Majority Dummy", "Stratified Dummy"]:
          model = get_base_models()[model_name]
        else:
          model = get_models()[model_name]

        pipeline = create_pipeline(
            model_name,
            model
            )

        # Apply tuned parameters
        pipeline.set_params(**best_param)

        # Train
        pipeline.fit(X_inner_train, y_inner_train)

        # Predict probabilities
        inner_probabilities[inner_valid_idx] = (
            pipeline.predict_proba(X_inner_valid)[:, 1]
        )

  ### Find best threshold ###
  thresholds=np.arange(0.01, 1.00, 0.01)
  best_threshold = 0.5
  best_mcc = -1
  for threshold in thresholds:
          y_pred = (inner_probabilities >= threshold).astype(int)
          mcc = matthews_corrcoef(y_train, y_pred)

          if mcc > best_mcc:
              best_mcc = mcc
              best_threshold = threshold

  return best_threshold, best_mcc

In [ ]:
def evaluate_outer_fold(estimator, X_test, y_test, threshold):
  test_probabilities = estimator.predict_proba(
        X_test
    )[:, 1]

  test_predictions = (
        test_probabilities >= threshold
    ).astype(int)

  ### Evaluate fold ###
  fold_metrics = evaluate_model(

        y_test= y_test,
        predictions= test_predictions,
        probabilities= test_probabilities

    )
  return fold_metrics, test_predictions, test_probabilities

In [ ]:
def run_outer_fold_loop(experiment1_ds, model_name, model, X, y, outer_cv, inner_cv,
                        scoring, groups=None, base_name=None):
  for fold, (train_idx, test_idx) in enumerate(
            outer_cv.split(X, y, groups),
            start=1):

        ################# Create train/test split #################
        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        if groups is None:
          group_train = None
        else:
          group_train = groups.iloc[train_idx]

        ################# Build pipeline #################
        pipeline = create_pipeline(
            model_name=model_name,
            estimator=model
        )


        ################# Get best pipeline and parameters #################
        best_pipeline, best_params, best_score = tune_model(
            model_name, pipeline, X_train, y_train,
            inner_cv, scoring, group_train)

        if base_name is None:
          base_name = model_name

        base_name = base_name.replace(" ","_")
        experiment1_ds[base_name]["best_params"].append(
            best_params
        )

        ################# Find best threshold #################
        threshold, best_inner_mcc = select_best_threshold(model_name, best_params, X_train, y_train,
                                                          inner_cv, group_train)
        experiment1_ds[base_name]["thresholds"].append(
            threshold
        )
        experiment1_ds[base_name]["best_inner_mcc"].append(
            best_inner_mcc
        )

        ################# Retrain on the outer training fold #################
        best_pipeline.fit(X_train, y_train)

        ################# Evaluate fold #################
        fold_metrics, y_pred, y_prob = evaluate_outer_fold(
            best_pipeline, X_test, y_test, threshold
        )

        fold_metrics['Fold'] = fold
        fold_metrics['Threshold'] = threshold
        experiment1_ds[base_name]["fold_metrics"].append(
            fold_metrics
        )

        ################# Save fold path #################
        fold_path = save_fold_predictions(FOLD_DIR, model_name, fold, best_pipeline, best_params, threshold,
                                          train_idx, test_idx, best_score, y_test, y_prob, y_pred)
        experiment1_ds[base_name]["saved_folds"].append(
            fold_path
        )

        ################# Save trained model #################
        model_dir = os.path.join(
          MODEL_DIR,
          base_name
        )

        os.makedirs(
            model_dir,
            exist_ok=True
        )
        model_path = os.path.join(model_dir, f"{base_name}_fold{fold}.joblib")

        joblib.dump(best_pipeline, model_path)
        experiment1_ds[base_name]["saved_models"].append(
            model_path
        )


In [ ]:
experiment1_ds2 = {}
result_rows = []

In [ ]:
OUTPUT_DIR = "/content/dataset2/age+diabetic_year_baseline"

MODEL_DIR = os.path.join(OUTPUT_DIR, "saved_models")
CSV_DIR = os.path.join(OUTPUT_DIR, "csv")
FOLD_DIR = os.path.join(OUTPUT_DIR, "fold_predictions")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(FOLD_DIR, exist_ok=True)

In [ ]:
for model_name, model in age_baseline.items():
  base_name = "A&D_LR"
  # print("=" * 70)
  print(base_name)
  # print("=" * 70)

  experiment1_ds2[base_name] = {
        "fold_metrics": [],
        "best_params": [],
        "thresholds": [],
        "best_inner_mcc": [],
        "saved_folds": [],
        "saved_models": []
    }
  age_features = ["Age", "Diabetic_Year"]

  X_age = ds2_X[age_features]

  run_outer_fold_loop(experiment1_ds2, model_name, model, X_age, ds2_y, outer_cv_d2,
                        inner_cv_d2, scoring="average_precision", groups=group, base_name=base_name)
  fold_df = pd.DataFrame(
        experiment1_ds2[base_name]["fold_metrics"]

  )
  experiment1_ds2[base_name]["fold_results"] = fold_df
  # Columns that should be averaged
  metric_columns = [
      "PR AUC",
      "ROC AUC",
      "F1",
      "MCC",
      "Sensitivity",
      "Specificity",
      "Balanced Accuracy",
      "Accuracy",
      "Brier Score"
  ]

  metric_df = fold_df[metric_columns]
  print(metric_df)
  mean = metric_df.mean()
  std = metric_df.std()

  experiment1_ds2[base_name]["mean"] = (
        mean.to_dict()
  )
  experiment1_ds2[base_name]["std"] = (
            std.to_dict()
  )

  csv_dir = os.path.join(
        CSV_DIR,
        base_name
  )

  os.makedirs(
        csv_dir,
        exist_ok=True
  )

  fold_df.to_csv(
      os.path.join(
          csv_dir,
          f"{base_name}_fold_results.csv"
      ),
      index=False
  )

  params_df = pd.DataFrame(
      experiment1_ds2[base_name]["best_params"]
  )

  params_df.insert(0, "Fold", range(1, 6))

  params_df.to_csv(
      os.path.join(
          csv_dir,
          f"{base_name}_best_parameters.csv"
      ),
      index=False
  )

  result_dict = {}
  result_dict["Model"] = f"(Age+Diabetic_Year) {model_name}"
  for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"

  result_rows.append(result_dict)

A&D_LR
     PR AUC   ROC AUC        F1       MCC  Sensitivity  Specificity  \
0  0.129670  0.621132  0.246057  0.177116     0.906977     0.368280   
1  0.150815  0.645727  0.217755  0.138696     0.855263     0.370166   
2  0.125449  0.676054  0.222997  0.185718     0.927536     0.388350   
3  0.126815  0.598012  0.199234  0.088867     0.712329     0.438472   
4  0.154295  0.549630  0.206897  0.076132     0.731707     0.389972   

   Balanced Accuracy  Accuracy  Brier Score  
0           0.637628  0.424096     0.240791  
1           0.612714  0.416250     0.247622  
2           0.657943  0.435443     0.243540  
3           0.575401  0.464103     0.244855  
4           0.560840  0.425000     0.092208  


In [ ]:
OUTPUT_DIR = "/content/dataset2"

MODEL_DIR = os.path.join(OUTPUT_DIR, "saved_models")
CSV_DIR = os.path.join(OUTPUT_DIR, "csv")
FOLD_DIR = os.path.join(OUTPUT_DIR, "fold_predictions")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(FOLD_DIR, exist_ok=True)

In [ ]:
for model_name, model in chain(base_models.items(), models.items()):

  # print("=" * 70)
  print(model_name)
  # print("=" * 70)
  base_name = model_name.replace(" ","_")
  experiment1_ds2[base_name] = {
      "fold_metrics": [],
      "best_params": [],
      "thresholds": [],
      "best_inner_mcc": [],
      "saved_folds": [],
      "saved_models": []
  }

  ################# Outer fold loop #################
  run_outer_fold_loop(experiment1_ds2, model_name, model, ds2_X, ds2_y, outer_cv_d2,
                        inner_cv_d2, scoring="average_precision", groups=group)

  ################# Add aggregation #################
  fold_df = pd.DataFrame(
        experiment1_ds2[base_name]["fold_metrics"]

  )
  experiment1_ds2[base_name]["fold_results"] = fold_df
  # Columns that should be averaged
  metric_columns = [
      "PR AUC",
      "ROC AUC",
      "F1",
      "MCC",
      "Sensitivity",
      "Specificity",
      "Balanced Accuracy",
      "Accuracy",
      "Brier Score"
  ]

  metric_df = fold_df[metric_columns]
  print(metric_df)
  mean = metric_df.mean()
  std = metric_df.std()

  experiment1_ds2[base_name]["mean"] = (
        mean.to_dict()
  )
  experiment1_ds2[base_name]["std"] = (
            std.to_dict()
  )

  csv_dir = os.path.join(
        CSV_DIR,
        base_name
  )

  os.makedirs(
        csv_dir,
        exist_ok=True
  )

  fold_df.to_csv(
      os.path.join(
          csv_dir,
          f"{base_name}_fold_results.csv"
      ),
      index=False,
      encoding='utf-8-sig'
  )

  params_df = pd.DataFrame(
      experiment1_ds2[base_name]["best_params"]
  )

  params_df.insert(0, "Fold", range(1, 6))

  params_df.to_csv(
      os.path.join(
          csv_dir,
          f"{base_name}_best_parameters.csv"
      ),
      index=False,
      encoding='utf-8-sig'
  )

  result_dict = {}
  result_dict["Model"] = model_name
  for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"

  result_rows.append(result_dict)

################# Summary #################
result_df = pd.DataFrame(result_rows)
print("=" * 70)
print("Summary")
print("=" * 70)
print(result_df)

result_df = result_df.sort_values("PR AUC", ascending=False)

result_df.to_csv(
    os.path.join(
        CSV_DIR,
        "experiment1_dataset2_result.csv"
    ),
    index=False,
    encoding='utf-8-sig'
)

joblib.dump(
    experiment1_ds2,
    os.path.join(OUTPUT_DIR, "experiment1_dataset2.joblib")
)

##### load joblib file #####
# experiment1_ds1 = joblib.load(
#     os.path.join(OUTPUT_DIR, "experiment1_dataset2.joblib")
# )

Majority Dummy
     PR AUC  ROC AUC   F1  MCC  Sensitivity  Specificity  Balanced Accuracy  \
0  0.103614      0.5  0.0  0.0          0.0          1.0                0.5   
1  0.095000      0.5  0.0  0.0          0.0          1.0                0.5   
2  0.087342      0.5  0.0  0.0          0.0          1.0                0.5   
3  0.093590      0.5  0.0  0.0          0.0          1.0                0.5   
4  0.102500      0.5  0.0  0.0          0.0          1.0                0.5   

   Accuracy  Brier Score  
0  0.896386     0.103614  
1  0.905000     0.095000  
2  0.912658     0.087342  
3  0.906410     0.093590  
4  0.897500     0.102500  
Stratified Dummy
     PR AUC   ROC AUC        F1       MCC  Sensitivity  Specificity  \
0  0.104119  0.502594  0.106509  0.005271     0.104651     0.900538   
1  0.097072  0.510177  0.115385  0.019894     0.118421     0.901934   
2  0.086351  0.492854  0.081081 -0.013450     0.086957     0.898752   
3  0.094869  0.506704  0.107383  0.013169     0

['/content/dataset2/experiment1_dataset2.joblib']